# 1 - Crear Spark Session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RetailStreamingLambda")
    .config(
        "spark.jars",
        "/opt/spark/jars/spark-sql-kafka-0-10_2.12-3.5.1.jar,/opt/spark/jars/spark-token-provider-kafka-0-10_2.12-3.5.1.jar,/opt/spark/jars/kafka-clients-3.6.1.jar,/opt/spark/jars/commons-pool2-2.11.1.jar"
    )
    .getOrCreate()
)

spark.version

/usr/local/lib/python3.8/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
26/08/07 20:57:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


'3.5.1'

In [2]:
!ls /opt/spark/jars/

commons-pool2-2.11.1.jar     spark-sql-kafka-0-10_2.12-3.5.1.jar
kafka-clients-3.6.1.jar      spark-token-provider-kafka-0-10_2.12-3.5.1.jar
mysql-connector-j-8.4.0.jar


# 2 - Verificar conexión con Kafka

In [3]:
kafka_server = "kafka:9092"

print(kafka_server)

kafka:9092


# 3 - Leer orders desde Kafka

In [4]:
orders_raw = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        kafka_server
    )
    .option(
        "subscribe",
        "orders_topic"
    )
    .option(
        "startingOffsets",
        "latest"
    )
    .load()
)

orders_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



# 4 - Convertir JSON de orders

In [5]:
from pyspark.sql.types import *

orders_schema = StructType([
    StructField(
        "order_id",
        IntegerType()
    ),
    StructField(
        "order_date",
        TimestampType()
    ),
    StructField(
        "order_customer_id",
        IntegerType()
    ),
    StructField(
        "order_status",
        StringType()
    )
])

### Trasformamos

In [6]:
from pyspark.sql.functions import *


orders_stream = (
    orders_raw
    .selectExpr(
        "CAST(value AS STRING) json"
    )
    .select(
        from_json(
            col("json"),
            orders_schema
        ).alias("data")
    )
    .select("data.*")
)


orders_stream.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)



# 5 - Mostrar streaming de orders

In [7]:
query_orders = (
    orders_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "truncate",
        False
    )
    .start()
)

26/08/07 20:57:40 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-ff91e59f-9389-4ede-b0a1-d4e712b9bf72. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/07 20:57:40 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+--------+----------+-----------------+------------+
|order_id|order_date|order_customer_id|order_status|
+--------+----------+-----------------+------------+
+--------+----------+-----------------+------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+--------+-------------------+-----------------+------------+
|order_id|order_date         |order_customer_id|order_status|
+--------+-------------------+-----------------+------------+
|69061   |2026-08-07 20:57:42|2742             |PROCESSING  |
+--------+-------------------+-----------------+------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+--------+-------------------+-----------------+------------+
|order_id|order_date         |order_customer_id|order_status|
+--------+-------------------+-----------------+------------+
|69062   |2026-08-07 20:57:47|6554             |CLOSED      |
+--------+-------------------+-----------------+------------+



In [8]:
query_orders.stop()

# 6 - Leer order_items

In [9]:
items_raw = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        kafka_server
    )
    .option(
        "subscribe",
        "order_items_topic"
    )
    .option(
        "startingOffsets",
        "latest"
    )
    .load()
)

In [10]:
items_schema = StructType([

    StructField(
        "order_item_id",
        IntegerType()
    ),

    StructField(
        "order_item_order_id",
        IntegerType()
    ),

    StructField(
        "order_item_product_id",
        IntegerType()
    ),

    StructField(
        "order_item_quantity",
        IntegerType()
    ),

    StructField(
        "order_item_subtotal",
        DoubleType()
    ),

    StructField(
        "order_item_product_price",
        DoubleType()
    )
])

In [11]:
items_stream = (
    items_raw
    .selectExpr(
        "CAST(value AS STRING) json"
    )
    .select(
        from_json(
            col("json"),
            items_schema
        ).alias("data")
    )
    .select("data.*")
)


items_stream.printSchema()

root
 |-- order_item_id: integer (nullable = true)
 |-- order_item_order_id: integer (nullable = true)
 |-- order_item_product_id: integer (nullable = true)
 |-- order_item_quantity: integer (nullable = true)
 |-- order_item_subtotal: double (nullable = true)
 |-- order_item_product_price: double (nullable = true)



# 7 - Mostrar items

In [12]:
query_items = (
    items_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "truncate",
        False
    )
    .start()
)

26/08/07 20:57:56 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-57e9d229-3835-4caf-8098-1f98f27532dc. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/07 20:57:56 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
|order_item_id|order_item_order_id|order_item_product_id|order_item_quantity|order_item_subtotal|order_item_product_price|
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+



[Stage 4:>                                                          (0 + 1) / 1]

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
|order_item_id|order_item_order_id|order_item_product_id|order_item_quantity|order_item_subtotal|order_item_product_price|
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+
|172714       |69064              |1127                 |3                  |102.0              |34.0                    |
|172715       |69064              |987                  |2                  |799.98             |399.99                  |
|172716       |69064              |438                  |2                  |189.98             |94.99                   |
+-------------+-------------------+---------------------+-------------------+-------------------+------------------------+



In [13]:
query_items.stop()

In [14]:
from pyspark.sql.functions import *
# "La columna order_date representa el tiempo del evento. Permite que lleguen datos atrasados hasta 10 minutos."
orders_stream_watermark = (
    orders_stream
    .withWatermark(
        "order_date",
        "2 minutes"
    )
)

In [15]:
#"Para este stream, considera válidos los eventos atrasados hasta 10 minutos usando event_time."
items_stream_watermark = (
    items_stream
    .withColumn(
        "event_time",
        current_timestamp()
    )
    .withWatermark(
        "event_time",
        "2 minutes"
    )
)

# Revisa un procesamiento en medio 

In [16]:
join_condition = (
    orders_stream_watermark.order_id ==
    items_stream_watermark.order_item_order_id
)

In [17]:
sales_stream = (
    orders_stream_watermark
    .join(
        items_stream_watermark,
        join_condition,
        "inner"
    )
)

In [18]:
ventas = (
    sales_stream
    .select(
        orders_stream_watermark.order_id,
        orders_stream_watermark.order_customer_id,
        orders_stream_watermark.order_status,
        items_stream_watermark.order_item_product_id,
        items_stream_watermark.order_item_quantity,
        items_stream_watermark.order_item_subtotal
    )
)

In [19]:
ventas.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_item_product_id: integer (nullable = true)
 |-- order_item_quantity: integer (nullable = true)
 |-- order_item_subtotal: double (nullable = true)



In [20]:
query_join = (
    ventas
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "truncate",
        False
    )
    .start()
)

26/08/07 20:58:12 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-843616cc-af07-4800-b4ff-baca73fca5b1. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/07 20:58:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
[Stage 7:=====>                                                  (20 + 2) / 200]

In [22]:
query_join.stop()

In [23]:
join_query = (
    ventas
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .trigger(
        processingTime="5 seconds"
    )
    .start()
)

26/08/07 20:58:30 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-b8204c88-5a6a-440b-851f-ae1c9bad060c. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/07 20:58:30 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+-----------------+------------+---------------------+-------------------+-------------------+
|order_id|order_customer_id|order_status|order_item_product_id|order_item_quantity|order_item_subtotal|
+--------+-----------------+------------+---------------------+-------------------+-------------------+
+--------+-----------------+------------+---------------------+-------------------+-------------------+



26/08/07 20:59:49 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 78964 milliseconds
[Stage 13:=====================================================>(198 + 2) / 200]

-------------------------------------------
Batch: 1
-------------------------------------------
+--------+-----------------+------------+---------------------+-------------------+-------------------+
|order_id|order_customer_id|order_status|order_item_product_id|order_item_quantity|order_item_subtotal|
+--------+-----------------+------------+---------------------+-------------------+-------------------+
|69073   |7100             |CLOSED      |1325                 |1                  |30.0               |
|69073   |7100             |CLOSED      |917                  |3                  |65.97              |
|69073   |7100             |CLOSED      |631                  |4                  |559.96             |
|69073   |7100             |CLOSED      |1255                 |4                  |159.96             |
|69077   |6830             |CLOSED      |711                  |2                  |99.98              |
|69077   |6830             |CLOSED      |1040                 |2       

26/08/07 21:00:57 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 68503 milliseconds
[Stage 16:======================>                                (80 + 2) / 200]

In [ ]:
ventas = (
    orders_stream_watermark
    .join(
        items_stream_watermark,
        orders_stream_watermark.order_id ==
        items_stream_watermark.order_item_order_id,
        "inner"
    )
)

streaming_query = (
    ventas
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "path",
        "hdfs://namenode:9000/lambda/speed"
    )
    .option(
        "checkpointLocation",
        "hdfs://namenode:9000/lambda/checkpoint_speed"
    )
    .trigger(
        processingTime="2 minutes"
    )
    .start()
)